# Retail Sales & Customer Analytics — SQL Measures

## Project overview

This notebook contains the SQL queries used to power key metrics and visualizations in the **Retail Sales & Customer Analytics Dashboard**, built with **Databricks SQL and AI/BI Dashboards**.

The queries focus on sales performance, customer purchasing behavior, product-category analysis, loyalty segmentation, and daily sales/order trends.

## Analytical scope

| Section | Analysis | Business value |
|---|---|---|
| 01 | Sales performance vs. target | Monitor progress toward the revenue goal |
| 02 | Customer sales details | Explore sales by customer, month, and product category |
| 03 | Product category & loyalty segment | Prepare data for analyzing revenue across product and loyalty dimensions |
| 04 | Daily sales & order volume | Compare daily revenue with order activity |

## Data sources

The queries reference the following tables in the Databricks catalog:

- `retail.sales`
- `retail.customers`
- `retail.sales_orders`

> **Note:** The SQL statements below retain the original logic from the exported Databricks notebook. Review the catalog and schema names, target value, and date filters if you reuse these queries in another environment or reporting period.


## 01. Sales Performance & Target Achievement

### Business question
How much revenue has the business generated, and how far is it from the sales target?

### Objective
Calculate the actual sales amount, compare it with a fixed sales target, and quantify the remaining revenue needed to reach that target.

### SQL logic
- `SUM(total_price)` calculates total sales.
- `3000000` defines the sales target.
- The target minus actual sales calculates the remaining amount.

### Expected output
| Column | Description |
|---|---|
| `actual_sales` | Total revenue generated |
| `sales_target` | Target revenue (3,000,000) |
| `remaining_to_target` | Difference between target and actual sales |

**Dashboard use:** Supports the Total Sales, Sales Target, and Remaining to Target KPIs.

> **Configuration note:** The target is hard-coded to 3,000,000 in this query. Update it when the business target changes.

In [ ]:
SELECT
    SUM(total_price) AS actual_sales,
    3000000 AS sales_target,
    3000000 - SUM(total_price) AS remaining_to_target
FROM learn_get_started_da_benlahcenhayat.retail.sales;

## 02. Customer Sales Analysis

### Business question
How are sales distributed across customers, months, and product categories?

### Objective
Retrieve customer-level sales records for the selected reporting period, including the customer name, transaction value, month, and product category.

### SQL logic
- Selects customer name and transaction sales value.
- Extracts the month from `order_date` using `date_format`.
- Filters records between 1 August 2019 and 31 October 2019.

### Expected output
| Column | Description |
|---|---|
| `customer_name` | Customer associated with the sale |
| `Total_Sales` | Sales value for the record |
| `Month` | Month extracted from the order date |
| `product_category` | Product category |

**Dashboard use:** Provides detailed data for customer and monthly sales exploration.

> **Reporting-period note:** The date range is fixed in the query. Adjust the dates when analyzing a different period.

In [ ]:
SELECT
  customer_name,
  total_price                       AS Total_Sales,
  date_format(order_date, 'MM')     AS Month,
  product_category
FROM learn_get_started_da_benlahcenhayat.retail.sales
WHERE order_date BETWEEN to_date('2019-08-01') AND to_date('2019-10-31');

## 03. Revenue by Product Category & Loyalty Segment

### Business question
How can sales records be analyzed by product category and customer loyalty segment?

### Objective
Combine sales and customer information so that each sales record can be associated with its product category and the customer's loyalty segment.

### SQL logic
- Uses the `sales` table as the transaction source.
- Joins the `customers` table using `customer_id`.
- Returns product category, loyalty segment, and transaction value.

### Expected output
| Column | Description |
|---|---|
| `product_category` | Category associated with the sale |
| `loyalty_segment` | Loyalty segment of the customer |
| `total_price` | Sales value for the record |

**Dashboard use:** Supplies the dimensions and sales values needed to analyze revenue across product categories and loyalty segments.

> **Data-grain note:** This query returns sales-level records; it does not aggregate revenue. Aggregation can be applied in the dashboard or in a separate SQL query.

In [ ]:
SELECT
  product_category,
  loyalty_segment,
  total_price
FROM learn_get_started_da_benlahcenhayat.retail.sales as s           -- the sales table
JOIN learn_get_started_da_benlahcenhayat.retail.customers as c         -- the customers table
  ON c.customer_id=s.customer_id;

## 04. Daily Sales & Order Volume Analysis

### Business question
How do daily sales revenue and order volume move across the days of a month?

### Objective
Calculate daily sales and daily order counts, then combine the two measures to support a comparison of revenue and order activity.

### SQL logic
- `sales_data` aggregates sales revenue by day extracted from `order_date`.
- `orders_data` derives the day from `order_datetime`, handling Unix timestamps stored as numeric values.
- The two datasets are joined using the day value.
- Results are sorted by day in ascending order.

### Expected output
| Column | Description |
|---|---|
| `day` | Day of the month |
| `total_sales` | Aggregated sales revenue for that day value |
| `total_orders` | Number of orders for that day value |

**Dashboard use:** Supports the daily sales and order-volume visualization.

> **Aggregation note:** The query groups by day of the month (`dd`) rather than the full calendar date. Consequently, the same day number across different months can be combined. The join also keeps only day values present in both datasets. Confirm that this matches the intended reporting period and chart grain before reusing the result.


In [ ]:
WITH sales_data AS (
  SELECT
    date_format(order_date, 'dd') AS day,
    SUM(total_price) AS total_sales
  FROM learn_get_started_da_benlahcenhayat.retail.sales         -- the sales table
  GROUP BY day
),
orders_data AS (
  SELECT
    CASE
      WHEN try_cast(sales_orders.order_datetime AS BIGINT) IS NOT NULL
      THEN DAY(FROM_UNIXTIME(sales_orders.order_datetime))
      ELSE NULL
    END AS day,
    COUNT(order_number) AS total_orders
  FROM learn_get_started_da_benlahcenhayat.retail.sales_orders        -- the sales_orders table
  GROUP BY day
)
SELECT
  cast(s.day AS INT) AS day,
  s.total_sales,
  o.total_orders
FROM sales_data s
JOIN orders_data o ON s.day = o.day
ORDER BY day asc; 